In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import sys
from pathlib import Path

# notebook 魔法：获取当前notebook工作目录
notebook_dir = Path.cwd()
# 向上一级作为项目根
project_root = notebook_dir.parent
sys.path.append(str(project_root))

%load_ext autoreload
%autoreload 2

# 解决中文、负号显示问题
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
df_raw = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = df_raw.copy()

print("原始数据集形状:",df.shape)
print('\n----------前5行----------')
display(df.head())

原始数据集形状: (7043, 21)

----------前5行----------


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [39]:
print('\n----------数据类型信息----------')
df.info()


----------数据类型信息----------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilli

In [40]:
# 1. 检查TotalCharges字段
# 1.1 查看TotalCharges字段的数据类型
print("TotalCharges数据类型:",df['TotalCharges'].dtype)
# 1.2 查看TotalCharges字段等于空字符串的行
empty_total = df[df['TotalCharges']==' ']
print(f"TotalCharges为空字符串的样本数量:{len(empty_total)}")
display(empty_total[['customerID','tenure','MonthlyCharges','TotalCharges']])
# 1.3 查看是否存在完全重复的行
print("完全重复行数:",df.duplicated().sum())
print("customerID是否唯一:",df['customerID'].nunique()==len(df))

TotalCharges数据类型: object
TotalCharges为空字符串的样本数量:11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


完全重复行数: 0
customerID是否唯一: True


In [41]:
from src.data_process import load_raw_data, binary_col_encode, preprocess_encode, build_engineered_features
# 2. 处理 TotalCharges,删除无关ID，标签映射
df_clean = load_raw_data(df)
# 校验
print("TotalCharges数据类型:",df_clean['TotalCharges'].dtype)
print("转换后TotalCharges等于0的样本数量:",(df_clean['TotalCharges']==0).sum())
display(df_clean[df_clean['TotalCharges']==0][['tenure','MonthlyCharges','TotalCharges']])

TotalCharges数据类型: float64
转换后TotalCharges等于0的样本数量: 11


,tenure,MonthlyCharges,TotalCharges
488,0,52.55,0.0
753,0,20.25,0.0
936,0,80.85,0.0
1082,0,25.75,0.0
1340,0,56.05,0.0
3331,0,19.85,0.0
3826,0,25.35,0.0
4380,0,20.00,0.0
5218,0,19.70,0.0
6670,0,73.35,0.0


In [42]:
# 3. 二分类特征 0-1 编码
binary_cols = ["gender","Partner","Dependents","PhoneService","PaperlessBilling"]
df_encoded = binary_col_encode(df_clean, cols=binary_cols)
print('\n----------二分类特征编码完成，查看样例----------')
print("二分类特征编码后的数据集形状:",df_encoded.shape)
display(df_encoded[binary_cols].head())


----------二分类特征编码完成，查看样例----------
二分类特征编码后的数据集形状: (7043, 20)


,gender,Partner,Dependents,PhoneService,PaperlessBilling
0,0,1,0,0,1
1,1,0,0,1,0
2,1,0,0,1,1
3,1,0,0,0,0
4,0,0,0,1,1


In [47]:
# 4. 多分类特征 OneHot 编码
# 4.1 基线模型-逻辑回归数据集
df_onehot_lr = preprocess_encode(df_encoded,drop_first=True)

print("onehot后的lr数据集形状:",df_onehot_lr.shape)
display(df_onehot_lr.head())
print('\n'+'-'*10+'查看新增列名'+'-'*10)
for c in df_onehot_lr.columns:
    print(c)

# 保存到output/processed下
df_onehot_lr.to_csv('../data/processed/lr_churn.csv',index=False,encoding='utf-8')
print('\n'+'-'*10+'lr预处理数据集已保存'+'-'*10)

onehot后的lr数据集形状: (7043, 31)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,0,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1,0,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,0,0,0,0,0,0,0,1,0



----------查看新增列名----------
gender
SeniorCitizen
Partner
Dependents
tenure
PhoneService
PaperlessBilling
MonthlyCharges
TotalCharges
Churn
MultipleLines_No phone service
MultipleLines_Yes
InternetService_Fiber optic
InternetService_No
OnlineSecurity_No internet service
OnlineSecurity_Yes
OnlineBackup_No internet service
OnlineBackup_Yes
DeviceProtection_No internet service
DeviceProtection_Yes
TechSupport_No internet service
TechSupport_Yes
StreamingTV_No internet service
StreamingTV_Yes
StreamingMovies_No internet service
StreamingMovies_Yes
Contract_One year
Contract_Two year
PaymentMethod_Credit card (automatic)
PaymentMethod_Electronic check
PaymentMethod_Mailed check

----------lr预处理数据集已保存----------


In [54]:
# 4.2 基线模型-逻辑回归(增加衍生特征)数据集
df_feature = df_encoded.copy()
df_feature_lr = build_engineered_features(df_feature)

print('\n'+'-'*10+'查看衍生特征样例'+'-'*10)
display(df_feature_lr[["tenure","MonthlyCharges","TotalCharges","avg_monthly_spend","user_life_stage","charge_ratio"]].head(10))

feature_cols = ["user_life_stage"]
df_feature_onehot_lr = preprocess_encode(df_feature_lr,drop_first=True,multi_cat_cols=feature_cols)

# 删掉和衍生特征高度冗余的原始列-tenure、MonthlyCharges
drop_cols = ["tenure", "MonthlyCharges"]
df_feature_onehot_lr = df_feature_onehot_lr.drop(columns=drop_cols).copy()
print("加入衍生特征onehot后的lr数据集形状:",df_feature_onehot_lr.shape)

df_feature_onehot_lr.to_csv('../data/processed/lr_feature_churn.csv',index=False,encoding='utf-8')
print('\n'+'-'*10+'加入衍生特征的lr预处理数据集已保存'+'-'*10)


----------查看衍生特征样例----------


,tenure,MonthlyCharges,TotalCharges,avg_monthly_spend,user_life_stage,charge_ratio
0,1,29.85,29.85,29.850000,new_user,1.000000
1,34,56.95,1889.50,55.573529,long_user,0.030140
2,2,53.85,108.15,54.075000,new_user,0.497920
3,45,42.30,1840.75,40.905556,long_user,0.022980
4,2,70.70,151.65,75.825000,new_user,0.466205
5,8,99.65,820.50,102.562500,mid_user,0.121450
6,22,89.10,1949.40,88.609091,mid_user,0.045706
7,10,29.75,301.90,30.190000,mid_user,0.098543
8,28,104.80,3046.05,108.787500,long_user,0.034405
9,62,56.15,3487.95,56.257258,long_user,0.016098


加入衍生特征onehot后的lr数据集形状: (7043, 33)

----------加入衍生特征的lr预处理数据集已保存----------


In [49]:
# 5. XGBoost模型数据集
# 5.1 基础数据集
df_onehot_XGBoost = preprocess_encode(df_encoded,drop_first=False)

print("onehot后的XGBoost数据集形状:",df_onehot_XGBoost.shape)
display(df_onehot_XGBoost.head())

# 保存到output/processed下
df_onehot_XGBoost.to_csv('../data/processed/XGBoost_churn.csv',index=False,encoding='utf-8')
print('\n'+'-'*10+'XGBoost预处理数据集已保存'+'-'*10)

onehot后的XGBoost数据集形状: (7043, 41)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,1,0,0,1,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,1,0,0,0,1,0,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,1,...,1,0,0,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1,0,0,0,1,0,1,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,1,0,0,1,0,0,0,0,1,0



----------XGBoost预处理数据集已保存----------


In [53]:
# 5.2 XGBoost(增加衍生特征)数据集
df_feature = df_encoded.copy()
df_feature_XGBoost = build_engineered_features(df_feature)

print('\n'+'-'*10+'查看衍生特征样例'+'-'*10)
display(df_feature_XGBoost[["tenure","MonthlyCharges","TotalCharges","avg_monthly_spend","user_life_stage","charge_ratio"]].head(10))

feature_cols = ["user_life_stage"]
df_feature_onehot_XGBoost = preprocess_encode(df_feature_XGBoost,drop_first=False,multi_cat_cols=feature_cols)

print("加入衍生特征onehot后的XGBoost数据集形状:",df_feature_onehot_XGBoost.shape)

df_feature_onehot_XGBoost.to_csv('../data/processed/XGBoost_feature_churn.csv',index=False,encoding='utf-8')
print('\n'+'-'*10+'加入衍生特征的XGBoost预处理数据集已保存'+'-'*10)


----------查看衍生特征样例----------


,tenure,MonthlyCharges,TotalCharges,avg_monthly_spend,user_life_stage,charge_ratio
0,1,29.85,29.85,29.850000,new_user,1.000000
1,34,56.95,1889.50,55.573529,long_user,0.030140
2,2,53.85,108.15,54.075000,new_user,0.497920
3,45,42.30,1840.75,40.905556,long_user,0.022980
4,2,70.70,151.65,75.825000,new_user,0.466205
5,8,99.65,820.50,102.562500,mid_user,0.121450
6,22,89.10,1949.40,88.609091,mid_user,0.045706
7,10,29.75,301.90,30.190000,mid_user,0.098543
8,28,104.80,3046.05,108.787500,long_user,0.034405
9,62,56.15,3487.95,56.257258,long_user,0.016098


加入衍生特征onehot后的XGBoost数据集形状: (7043, 46)

----------加入衍生特征的XGBoost预处理数据集已保存----------
